# Gold: star schema for BI (Silver -> dimensions + facts)
Silver tables are clean but shaped like the source files. Gold reshapes them into a **star schema**: small **dimension** tables (who / what / where / when) and **fact** tables (the numbers), joined by whole-number keys. Power BI reads these tables.

| Fact table (grain) | Dimensions | Answers business question |
|---|---|---|
| `fact_quintile_weekly` (quintile x week) | `dim_date`, `dim_quintile` | 1. Who is hurting most (Q1 vs Q5 gap) |
| `fact_wfp_price_monthly` (market x commodity x month) | `dim_date`, `dim_market`, `dim_commodity` | 2. Regional volatility of staple foods |
| `fact_item_weekly` (item x week) | `dim_date`, `dim_item` | 3. Fuel to food pass-through, 4. Subsidy scenario |
| `fact_city_price_weekly` (item x city x week) | `dim_date`, `dim_item`, `dim_city` | City price spread |

**How it works:** (1) build every dimension from Silver, (2) build every fact (keys are made from the natural key with a hash, so the same item always gets the same key), (3) `MERGE INTO` each table: insert new rows, update changed rows, leave the rest, (4) write one row per table to `pipeline_execution_logs` (rows aggregated, inserted, updated, start and end time), (5) `OPTIMIZE ... ZORDER BY date_key` on the fact tables.

`week_ending_dates`: `all` (every week in Silver), `none` (skip the PBS facts) or a list like `24.09.2026,01.10.2026`. Dimensions are always rebuilt from all of Silver (they are small). WFP fact: a full load merges everything, an incremental load only the last `lookback_months`.

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full_backfill"])
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("week_ending_dates", "all")        # all / none / 24.09.2026,01.10.2026
dbutils.widgets.text("lookback_months", "3")
dbutils.widgets.dropdown("optimize_tables", "yes", ["yes", "no"])

In [0]:
# ---- settings and small helpers (the same block is in every notebook) ----
import json, os, re, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

CATALOG = "workspace"      # change these two lines if your catalog / schema have other names
SCHEMA = "default"

def T(name):
    """Full table name, e.g. T("silver_quintile_index") -> workspace.default.silver_quintile_index"""
    return f"{CATALOG}.{SCHEMA}.{name}"

AUDIT_TABLE = T("pipeline_execution_logs")
QUARANTINE_TABLE = T("quarantine_records")

def now():
    """Current time (UTC)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)

def sql_text(text):
    """Makes a value safe inside a SQL string: O'Brien -> O''Brien"""
    return str(text).replace("'", "''")

def norm_date(text):
    """Turns 1.10.2026, 01-10-2026 or 2026-10-01 into 01.10.2026"""
    for pattern in ["%d.%m.%Y", "%d-%m-%Y", "%Y-%m-%d"]:
        try:
            return datetime.strptime(str(text).strip(), pattern).strftime("%d.%m.%Y")
        except ValueError:
            pass
    raise ValueError(f"Cannot read the date '{text}'. Use dd.mm.yyyy, for example 01.10.2026")

# ---- execution log: one row per layer, per file/week, per run ----
AUDIT_COLUMNS = ("log_id string, batch_id string, load_type string, layer string, file_processed string, "
                 "target_table string, start_time timestamp, end_time timestamp, status string, "
                 "rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, "
                 "error_message string, message string")

def log_audit(batch_id, load_type, layer, file_processed, target_table, start_time, status,
              rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0, message=""):
    """status is Success, Warning (success with notes), Skipped or Failure."""
    message = (message or "")[:1500]
    error_message = message if status == "Failure" else None      # error_message is only filled for failures
    row = (str(uuid.uuid4()), batch_id, load_type, layer, str(file_processed), target_table,
           start_time, now(), status, int(rows_read), int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, message)
    spark.createDataFrame([row], AUDIT_COLUMNS).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# ---- quarantine: rows (or whole files) that failed a check, saved together with the reason ----
def quarantine_from_list(items):
    """items = list of (record_ref, reason, record_json)"""
    return spark.createDataFrame(items, "record_ref string, reason string, record_json string")

def save_quarantine(df, layer, source_key, batch_id):
    """Replaces the earlier quarantine rows of the same layer and file, so re-runs never pile up duplicates.
    df needs the columns record_ref, reason and record_json."""
    spark.sql(f"DELETE FROM {QUARANTINE_TABLE} WHERE layer = '{layer}' AND source_file = '{sql_text(source_key)}'")
    rows = df.select(F.expr("uuid()").alias("quarantine_id"), F.lit(batch_id).alias("batch_id"),
                     F.lit(layer).alias("layer"), F.lit(source_key).alias("source_file"),
                     "record_ref", "reason", "record_json", F.current_timestamp().alias("quarantined_at"))
    rows.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)


In [0]:
from pyspark.sql.window import Window

LAYER = "Silver-to-Gold"

SILVER_QUINTILE = T("silver_quintile_index")
SILVER_WEIGHTS = T("silver_item_weight_impact")
SILVER_CITY = T("silver_city_price_observation")
SILVER_WFP = T("silver_wfp_price_observation")

DIM_DATE, DIM_QUINTILE, DIM_ITEM = T("dim_date"), T("dim_quintile"), T("dim_item")
DIM_CITY, DIM_MARKET, DIM_COMMODITY = T("dim_city"), T("dim_market"), T("dim_commodity")
FACT_QUINTILE, FACT_ITEM = T("fact_quintile_weekly"), T("fact_item_weekly")
FACT_CITY, FACT_WFP = T("fact_city_price_weekly"), T("fact_wfp_price_monthly")

# ---- keys ----
def make_key(*columns):
    """Surrogate key: a whole number made from the natural key (e.g. the item name).
    The same item always gets the same number, so a MERGE can run again and again without creating new keys."""
    return F.pmod(F.hash(*columns), F.lit(2147483647))

def date_key(column):
    """2026-09-24 -> 20260924"""
    return F.date_format(F.col(column), "yyyyMMdd").cast("int")

def stamp(df):
    """Adds the load_timestamp column."""
    return df.withColumn("load_timestamp", F.current_timestamp())

def fix_item_names(df):
    """Fixes known typos in the PBS item names, so one item has one name (and one key) in every week.
    Add more rules here if you find other typos, for example: .withColumn("item_name", F.regexp_replace("item_name", r"^Wrong", "Right"))"""
    return df.withColumn("item_name", F.regexp_replace("item_name", r"^Gad Charges", "Gas Charges"))

# ---- dimensions ----
FUEL_WORDS = r"\b(petrol|diesel|lpg|kerosene|electricity|gas|firewood|fuel)\b"

def build_dim_date(first_day, last_day):
    """One row per calendar day between the first and the last date in Silver."""
    days = spark.sql(f"SELECT explode(sequence(DATE'{first_day}', DATE'{last_day}', interval 1 day)) AS full_date")
    return stamp(days.select(
        date_key("full_date").alias("date_key"), "full_date",
        F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"), F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"), F.date_format("full_date", "yyyy-MM").alias("year_month"),
        F.weekofyear("full_date").alias("week_of_year"), F.date_format("full_date", "EEEE").alias("day_name")))

def build_dim_quintile(silver_quintile):
    """Q1..Q5 and Combined. income_band is the latest band published. 'Lowest' in the weights table = Q1."""
    latest = silver_quintile.groupBy("quintile").agg(F.max_by("income_band", "week_ending_date").alias("income_band"))
    rank = F.when(F.col("quintile") == "Combined", 6).otherwise(F.regexp_extract("quintile", r"Q(\d)", 1).cast("int"))
    group = F.when(F.col("quintile") == "Q1", "Lowest").when(F.col("quintile") == "Combined", "Combined")
    return stamp(latest.select(make_key("quintile").alias("quintile_key"), "quintile", "income_band",
                               rank.alias("income_rank"), group.alias("weight_group")))

def build_dim_item(silver_weights, silver_city):
    """Every item seen in the weight table or the city price table, with a Fuel & Energy / Food & Other group."""
    items = silver_weights.select("item_name", "unit").unionByName(silver_city.select("item_name", "unit"))
    items = items.groupBy("item_name").agg(F.max("unit").alias("unit"))
    group = F.when(F.lower(F.col("item_name")).rlike(FUEL_WORDS), "Fuel & Energy").otherwise("Food & Other")
    return stamp(items.select(make_key("item_name").alias("item_key"), "item_name", "unit", group.alias("item_group")))

def build_dim_city(silver_city):
    cities = silver_city.groupBy("city_code").agg(F.max("city").alias("city"))
    return stamp(cities.select(make_key("city_code").alias("city_key"), "city", "city_code"))

def build_dim_market(silver_wfp):
    markets = silver_wfp.groupBy("country", "market").agg(F.max("admin_region").alias("admin_region"))
    return stamp(markets.select(make_key("country", "market").alias("market_key"), "country", "admin_region", "market"))

def build_dim_commodity(silver_wfp):
    commodities = silver_wfp.groupBy("commodity_name", "unit").agg(F.max("category").alias("category"))
    return stamp(commodities.select(make_key("commodity_name", "unit").alias("commodity_key"),
                                    "commodity_name", "unit", "category"))

# ---- facts ----
def build_fact_quintile(silver_quintile):
    """One row per quintile per week. index_vs_combined = the group's index minus the overall (Combined) index."""
    same_week = Window.partitionBy("week_ending_date")
    combined_index = F.max(F.when(F.col("quintile") == "Combined", F.col("index_value"))).over(same_week)
    return stamp(silver_quintile.select(
        date_key("week_ending_date").alias("date_key"), make_key("quintile").alias("quintile_key"),
        "index_value", "index_prev_week", "index_prev_year", "pct_change_wow", "pct_change_yoy",
        (F.col("index_value") - combined_index).alias("index_vs_combined")))

def build_fact_item(silver_weights):
    """One row per item per week: price, price change, and the item's weight and impact for Lowest and Combined."""
    return stamp(silver_weights.select(
        date_key("week_ending_date").alias("date_key"), make_key("item_name").alias("item_key"), "price_direction",
        "price_current", "price_prev_week", "price_prev_year", "pct_change_wow", "pct_change_yoy",
        "weight_lowest", "weight_combined", "impact_lowest", "impact_combined"))

def build_fact_city(silver_city):
    """One row per item per city per week, with the % change of the average price vs the week before."""
    same_series = Window.partitionBy("item_name", "city_code").orderBy("week_ending_date")
    previous = F.lag("avg_price").over(same_series)
    wow = F.when(previous > 0, (F.col("avg_price") - previous) / previous * 100)
    return stamp(silver_city.select(
        date_key("week_ending_date").alias("date_key"), make_key("item_name").alias("item_key"),
        make_key("city_code").alias("city_key"), "min_price", "avg_price", "max_price", "price_dispersion_pct",
        wow.alias("avg_price_wow_pct")))

def build_fact_wfp(silver_wfp):
    """One row per market, commodity, price type and month, with the % change vs the month before."""
    same_series = Window.partitionBy("country", "market", "commodity_name", "unit", "price_type").orderBy("price_date")
    previous_price = F.lag("price").over(same_series)
    previous_date = F.lag("price_date").over(same_series)
    exactly_one_month = (F.months_between(F.col("price_date"), previous_date) == 1) & (previous_price > 0)
    mom = F.when(exactly_one_month, (F.col("price") - previous_price) / previous_price * 100)
    return stamp(silver_wfp.select(
        date_key("price_date").alias("date_key"), make_key("country", "market").alias("market_key"),
        make_key("commodity_name", "unit").alias("commodity_key"), "price_type", "price", "currency",
        mom.alias("pct_change_mom"), "is_revised_flag"))

# ---- save: MERGE INTO ----
def merge_into(df, target, key_columns, compare_columns):
    """Insert new rows, update rows whose values changed, leave the rest. Returns (inserted, updated)."""
    df.createOrReplaceTempView("gold_new_rows")
    same_key = " AND ".join(f"t.{c} <=> s.{c}" for c in key_columns)
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in compare_columns)
    spark.sql(f"""
        MERGE INTO {target} t USING gold_new_rows s ON {same_key}
        WHEN MATCHED AND ({changed}) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")
    history = (spark.sql(f"DESCRIBE HISTORY {target}").filter("operation = 'MERGE'")
                    .orderBy(F.col("version").desc()).limit(1).collect())
    if len(history) == 0:
        return 0, 0
    metrics = history[0]["operationMetrics"]
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))

In [0]:
load_type = dbutils.widgets.get("load_type")
batch_id = dbutils.widgets.get("batch_id").strip() or f"manual_{now():%Y%m%dT%H%M%S}"
lookback_months = int(dbutils.widgets.get("lookback_months") or 3)
weeks_text = dbutils.widgets.get("week_ending_dates").strip().lower()
failures = []

# which PBS weeks to refresh: None = all weeks, [] = none, or a list of date keys
if weeks_text in ("", "all"):
    week_keys = None
elif weeks_text == "none":
    week_keys = []
else:
    week_keys = [int(datetime.strptime(norm_date(w), "%d.%m.%Y").strftime("%Y%m%d")) for w in weeks_text.split(",") if w.strip()]

def only_selected_weeks(df):
    if week_keys is None:
        return df
    if len(week_keys) == 0:
        return df.filter(F.lit(False))
    return df.filter(F.col("date_key").isin(week_keys))

def refresh_table(target, df, key_columns, compare_columns):
    """MERGE one Gold table and write its row in the execution log."""
    start = now()
    name = target.split(".")[-1]
    try:
        rows = df.count()                                   # rows aggregated from Silver
        if rows == 0:
            log_audit(batch_id, load_type, LAYER, name, target, start, "Skipped", message="no rows to process")
            print(f"Skipped  {name}: no rows to process")
            return
        inserted, updated = merge_into(df, target, key_columns, compare_columns)
        message = f"{rows:,} rows aggregated; inserted {inserted:,}, updated {updated:,}, unchanged {max(rows - inserted - updated, 0):,}"
        log_audit(batch_id, load_type, LAYER, name, target, start, "Success",
                  rows_read=rows, rows_inserted=inserted, rows_updated=updated, message=message)
        print(f"Success  {name}: {message}")
    except Exception as e:
        log_audit(batch_id, load_type, LAYER, name, target, start, "Failure", message=str(e))
        print(f"FAILED   {name}: {str(e)[:300]}")
        failures.append(f"{name}: {str(e)[:200]}")

# ---- read Silver ----
silver_quintile = spark.table(SILVER_QUINTILE)
silver_weights = fix_item_names(spark.table(SILVER_WEIGHTS))      # item names with known typos fixed
silver_city = fix_item_names(spark.table(SILVER_CITY))
silver_wfp = spark.table(SILVER_WFP)

firsts = [silver_quintile.agg(F.min("week_ending_date")).collect()[0][0], silver_wfp.agg(F.min("price_date")).collect()[0][0]]
lasts = [silver_quintile.agg(F.max("week_ending_date")).collect()[0][0], silver_wfp.agg(F.max("price_date")).collect()[0][0]]
first_day = min(d for d in firsts if d is not None)
last_day = max(d for d in lasts if d is not None)

# ---- 1) dimensions (always rebuilt from all of Silver: they are small) ----
refresh_table(DIM_DATE, build_dim_date(first_day, last_day), ["date_key"],
              ["full_date", "year", "quarter", "month", "month_name", "year_month", "week_of_year", "day_name"])
refresh_table(DIM_QUINTILE, build_dim_quintile(silver_quintile), ["quintile_key"],
              ["quintile", "income_band", "income_rank", "weight_group"])
refresh_table(DIM_ITEM, build_dim_item(silver_weights, silver_city), ["item_key"], ["item_name", "unit", "item_group"])
refresh_table(DIM_CITY, build_dim_city(silver_city), ["city_key"], ["city", "city_code"])
refresh_table(DIM_MARKET, build_dim_market(silver_wfp), ["market_key"], ["country", "admin_region", "market"])
refresh_table(DIM_COMMODITY, build_dim_commodity(silver_wfp), ["commodity_key"], ["commodity_name", "unit", "category"])

# ---- 2) PBS facts (only the selected weeks) ----
refresh_table(FACT_QUINTILE, only_selected_weeks(build_fact_quintile(silver_quintile)), ["date_key", "quintile_key"],
              ["index_value", "index_prev_week", "index_prev_year", "pct_change_wow", "pct_change_yoy", "index_vs_combined"])
refresh_table(FACT_ITEM, only_selected_weeks(build_fact_item(silver_weights)), ["date_key", "item_key"],
              ["price_direction", "price_current", "price_prev_week", "price_prev_year", "pct_change_wow",
               "pct_change_yoy", "weight_lowest", "weight_combined", "impact_lowest", "impact_combined"])
refresh_table(FACT_CITY, only_selected_weeks(build_fact_city(silver_city)), ["date_key", "item_key", "city_key"],
              ["min_price", "avg_price", "max_price", "price_dispersion_pct", "avg_price_wow_pct"])

# ---- 3) WFP fact: full load = everything, incremental = only the last few months ----
fact_wfp = build_fact_wfp(silver_wfp)        # the month-on-month change needs the whole history, so filter AFTER building
if load_type == "incremental" and spark.catalog.tableExists(FACT_WFP):
    latest_key = spark.table(FACT_WFP).agg(F.max("date_key")).collect()[0][0]
    if latest_key is not None:
        cutoff = spark.sql(f"SELECT CAST(date_format(add_months(to_date('{latest_key}', 'yyyyMMdd'), -{lookback_months}), 'yyyyMMdd') AS INT)").collect()[0][0]
        fact_wfp = fact_wfp.filter(F.col("date_key") >= cutoff)
refresh_table(FACT_WFP, fact_wfp, ["date_key", "market_key", "commodity_key", "price_type"],
              ["price", "currency", "pct_change_mom", "is_revised_flag"])

Success  dim_date: 8,317 rows aggregated; inserted 0, updated 0, unchanged 8,317
Success  dim_quintile: 6 rows aggregated; inserted 0, updated 0, unchanged 6
Success  dim_item: 53 rows aggregated; inserted 53, updated 0, unchanged 0
Success  dim_city: 17 rows aggregated; inserted 0, updated 0, unchanged 17
Success  dim_market: 5 rows aggregated; inserted 0, updated 0, unchanged 5
Success  dim_commodity: 17 rows aggregated; inserted 0, updated 0, unchanged 17
Success  fact_quintile_weekly: 288 rows aggregated; inserted 0, updated 0, unchanged 288
Success  fact_item_weekly: 2,448 rows aggregated; inserted 2,448, updated 0, unchanged 0
Success  fact_city_price_weekly: 41,472 rows aggregated; inserted 41,472, updated 0, unchanged 0
Success  fact_wfp_price_monthly: 9,342 rows aggregated; inserted 0, updated 0, unchanged 9,342


In [0]:
# ---- 4) speed up reads: compact the files and cluster them by date_key ----
if dbutils.widgets.get("optimize_tables") == "yes":
    for table in [FACT_QUINTILE, FACT_ITEM, FACT_CITY, FACT_WFP]:
        try:
            display(spark.sql(f"OPTIMIZE {table} ZORDER BY (date_key)"))
        except Exception as e:
            print(f"OPTIMIZE skipped for {table}: {str(e)[:200]}")

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 11527), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1792950017183, 1792950018030, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null, 0)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 70270), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1792950019826, 1792950020444, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null, 0)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 193836), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1792950022040, 1792950022654, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null, 0)"


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 81399), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1792950023709, 1792950024335, 8, 0, null, List(0, 0), null, 9, 9, 0, 0, null, null, 0)"


In [0]:
# ---- 5) the query plan of the most complex fact build (window + shuffle), for the architecture report ----
build_fact_wfp(spark.table(SILVER_WFP)).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Initial Plan ==
   PhotonResultStage (6)
   +- PhotonColumnarToRow (5)
      +- PhotonWindow (4)
         +- PhotonSort (3)
            +- PhotonProject (2)
               +- PhotonScan parquet workspace.default.silver_wfp_price_observation (1)


(1) PhotonScan parquet workspace.default.silver_wfp_price_observation
Output [9]: [country#24468, market#24470, commodity_name#24471, unit#24473, price_type#24474, price#24475, currency#24476, price_date#24477, is_revised_flag#24478]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-injf5/uc/9e460a75-65fb-4137-93df-a5322a2816af/f9bb2ecb-be8e-4d8f-8391-94d52a9a3285/__unitystorage/catalogs/3fb3eb5e-d3f2-4ced-a573-8644042903f2/tables/43150220-668a-479a-b772-c1097b8b9691]
ReadSchema: struct<country:string,market:string,commodity_name:string,unit:string,price_type:string,price:double,currency:string,price_date:date,is_revised_flag:boolean>

(2) PhotonProject
Input [9]: [country#24468, market#24470

In [0]:
if len(failures) > 0:
    raise Exception("Gold refresh failed for: " + " | ".join(failures))
dbutils.notebook.exit(json.dumps({"status": "Success", "batch_id": batch_id}))